In [ ]:
from sklearn import metrics
import scanpy as sc
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
import os

from concurrent.futures import ProcessPoolExecutor, as_completed

from load_data import multiHIVE_data, DATASET_SEEDS_PAIRS

def _evaluate_clustering(labels_true, labels_pred):
    results = {
        'normalized_mutual_info': metrics.normalized_mutual_info_score(labels_true, labels_pred),
        'adjusted_rand_index': metrics.adjusted_rand_score(labels_true, labels_pred),
        'fowlkes_mallows': metrics.fowlkes_mallows_score(labels_true, labels_pred),
    }
    return results

def _compute_single_resolution(dataset, res, seed):
    """Helper function to compute clustering metrics for a single resolution."""
    adata = multiHIVE_data(dataset, seed)
    sc.pp.neighbors(adata, use_rep='latent')
    sc.tl.leiden(adata, resolution=res, key_added="Z_leiden")
    y_pred = adata.obs['Z_leiden'].values
    y_true = adata.obs['cell_type'].values
    result = _evaluate_clustering(y_true, y_pred)
    return res, result

def compute_clustering_metrics(dataset, seed, re_calculate=False): # parallel wrt resolution
    try:
        if not re_calculate and os.path.exists(f'./Results/{dataset}/{seed}_bio_metrics.csv'):
            print(f"Metrics for {dataset} - {seed} already exist. Skipping computation.")
            return

        os.makedirs(f'./Results/{dataset}/', exist_ok=True)
        
        resolutions = np.linspace(0.1, 2, 20)
        results = {}
        
        # Parallelize across resolutions
        with ProcessPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(_compute_single_resolution, dataset, res, seed) for res in resolutions]
            for future in as_completed(futures):
                res, result = future.result()
                results[res] = result
        
        results = pd.DataFrame(results).T
        results.to_csv(f'./Results/{dataset}/{seed}_bio_metrics.csv')
    except Exception as e:
        print(f"Error processing {dataset} - {seed}: {str(e)}")

In [ ]:
def compute_clustering_metrics(dataset, seed, re_calculate=False): # seq wrt resolution
    try:
        if not re_calculate and os.path.exists(f'./Results/{dataset}/{seed}_bio_metrics.csv'):
            print(f"Metrics for {dataset} - {seed} already exist. Skipping computation.")
            return

        os.makedirs(f'./Results/{dataset}', exist_ok=True)
        adata = multiHIVE_data(dataset, seed)
        results = {}
        sc.pp.neighbors(adata, use_rep='latent')
        for res in np.linspace(0.1, 2, 20):
            sc.tl.leiden(adata, resolution=res, key_added="Z_leiden")
            y_pred = adata.obs['Z_leiden'].values
            y_true = adata.obs['cell_type'].values
            result = _evaluate_clustering(y_true, y_pred)
            results[res] = result
        results = pd.DataFrame(results).T
        results.to_csv(f'./Results/{dataset}/{seed}_bio_metrics.csv')
    except Exception as e:
        print(f"Error processing {dataset} - {seed}: {str(e)}")

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_pair(args):
    dataset, seed = args
    compute_clustering_metrics(dataset, seed)
    return dataset, seed

with ProcessPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_pair, pair) for pair in DATASET_SEEDS_PAIRS]
    for future in tqdm(as_completed(futures), total=len(DATASET_SEEDS_PAIRS)):
        dataset, seed = future.result()